In [4]:
import pandas as pd

all_assays = pd.read_csv('../../HTS_data/output/all_assays_merged.csv')

print("\033[1mSelect an assay from the following list:\033[0m")
for assay in all_assays.columns.tolist()[1:]:
    print("  " + assay)

Select an assay from the following list:
  TOX21_AChE_Colorimetric_Antagonist
  TOX21_AP1_BLA_Agonist_ratio
  TOX21_ARE_BLA_Agonist_ratio
  TOX21_AR_BLA_Agonist_ratio
  TOX21_AR_BLA_Antagonist_ratio
  TOX21_AR_LUC_MDAKB2_Antagonist_0_5nM_R1881
  TOX21_AR_LUC_MDAKB2_Antagonist_10nM_R1881
  TOX21_AhR_LUC_Agonist
  TOX21_Aromatase_LUC_Antagonist
  TOX21_CAR_LUC_Agonist
  TOX21_DT40_100_LUC
  TOX21_DT40_657_LUC
  TOX21_DT40_LUC
  TOX21_ERR_LUC_Antagonist
  TOX21_ERa_BLA_Agonist_ratio
  TOX21_ERa_BLA_Antagonist_ratio
  TOX21_ERa_LUC_VM7_Agonist
  TOX21_ERa_LUC_VM7_Antagonist_0_1nM_E2
  TOX21_ERa_LUC_VM7_Antagonist_0_5nM_E2
  TOX21_ERb_BLA_Antagonist_ratio
  TOX21_FXR_BLA_Antagonist_ratio
  TOX21_GR_BLA_Agonist_ratio
  TOX21_GR_BLA_Antagonist_ratio
  TOX21_HDAC_LUC_Antagonist
  TOX21_MMP_ratio
  TOX21_PGC_ERR_LUC_Antagonist
  TOX21_PPARg_BLA_Antagonist_ratio
  TOX21_PR_BLA_Antagonist_ratio
  TOX21_PXR_LUC_Agonist
  TOX21_RAR_LUC_Antagonist
  TOX21_RORg_LUC_CHO_Antagonist
  TOX21_SBE_BLA_Anta

In [ ]:
assay_name = None  # Replace with the desired assay name from the printed list
assay_name = "TOX21_p450_CYP2C9_Antagonist"


# ------------------------------ imports -------------------------------------- #
import numpy as np
import pandas as pd
from datetime import datetime
from collections import Counter
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, auc,
    accuracy_score, f1_score, confusion_matrix,
    average_precision_score, brier_score_loss, 
    log_loss)
from sklearn.ensemble import RandomForestClassifier
from boruta import BorutaPy
import xgboost as xgb
from pathlib import Path
import json
import re
import joblib
from imblearn.over_sampling import SMOTENC
from sklearn.model_selection import RandomizedSearchCV
import shap
from imblearn.pipeline import Pipeline as ImbPipeline


# ------------------------------ config --------------------------------------- #
OUT_BASE = Path("../output")
OUT_BASE.mkdir(parents=True, exist_ok=True)

# Configuration
N_SPLITS = 3
RANDOM_STATE = 21
N_JOBS = 40
STABILITY_RUNS = 5
STABILITY_MIN_FRAC = 0.6
SAVE_MODELS = False  # Set to True if you need model files for predictions
SAVE_DETAILED_SHAP = False  # Set to True only if you need per-fold SHAP values

# Initialize storage
fold_importances = []
fold_metrics_detailed = []
fold_selected_features = []
fold_class_distributions = []
shap_data_per_fold = []  # For OOF SHAP aggregation


# ------------------------------ data ----------------------------------------- #
chemical_httr_assay_aggregated = pd.read_feather("../data/chemical_httr_assay_aggregated.feather")

# Identify feature columns
httr_cols = [col for col in chemical_httr_assay_aggregated.columns 
             if col.endswith(('_max', '_dose_at_max', '_AUC_neg'))]
chemical_cols = [col for col in chemical_httr_assay_aggregated.columns if col.startswith('MACCS_')]
assay_cols = [col for col in chemical_httr_assay_aggregated.columns if col.startswith('TOX21_')]

selected_assay = assay_name

# Remove cell_type column if present
if 'cell_type' in chemical_httr_assay_aggregated.columns:
    chemical_httr_assay_aggregated = chemical_httr_assay_aggregated.drop(columns=['cell_type'])

# Filter to samples with target assay data
df_filtered = chemical_httr_assay_aggregated.dropna(subset=[selected_assay]).copy()

# Keep only the selected assay and drop other assays
cols_to_drop = [c for c in df_filtered.columns if c.startswith("TOX21_") and c != selected_assay]
if 'chnm' in df_filtered.columns:
    cols_to_drop.append('chnm')
df_filtered.drop(columns=cols_to_drop, inplace=True)

# Setup output directory
ASSAY_SAFE = re.sub(r"[^A-Za-z0-9_.-]+", "_", selected_assay)
RUN_DIR = OUT_BASE / ASSAY_SAFE
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Identify metadata columns to keep
IDs_TO_KEEP = [c for c in ['outcome_id', 'epa_sample_id'] if c in df_filtered.columns]

# Prepare target and grouping variables
y_raw = df_filtered[selected_assay].astype(int).values
groups = df_filtered['outcome_id'].astype(str).values

# Calculate initial statistics
pos = int((y_raw == 1).sum())
neg = int((y_raw == 0).sum())

print(f"\nTarget: \033[1m{selected_assay}\033[0m")
print(f"Samples: {len(df_filtered):,} | Positives: {pos:,} | Negatives: {neg:,}")
print(f"Class ratio: {pos/neg:.3f}\n")

initial_stats = {
    'total_samples': len(chemical_httr_assay_aggregated),
    'samples_with_target': len(df_filtered),
    'n_httr_features': len(httr_cols),
    'n_chemical_features': len(chemical_cols),
    'n_assay_features': len(assay_cols),
    'target_positives': pos,
    'target_negatives': neg,
    'target_balance_ratio': min(pos, neg) / max(pos, neg),
    'unique_chemicals': df_filtered['outcome_id'].nunique() if 'outcome_id' in df_filtered.columns else None
}


# ------------------------------ CV setup ------------------------------------- #
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

feature_hits = Counter()
shap_values_all_folds = []

start_all = datetime.now()

# OOF predictions
oof_proba = np.full(len(df_filtered), np.nan)
oof_true = np.zeros(len(df_filtered), dtype=int)


# ------------------------------ CV loop -------------------------------------- #
for fold, (train_idx, test_idx) in enumerate(cv.split(df_filtered, y_raw, groups), start=1):
    
    train = df_filtered.iloc[train_idx].reset_index(drop=True)
    test = df_filtered.iloc[test_idx].reset_index(drop=True)
    
    y_train = train[selected_assay].astype(int).values
    y_test = test[selected_assay].astype(int).values
    
    # Safety check
    if len(np.unique(y_train)) < 2:
        print(f"[Fold {fold}] skipped (train has a single class).")
        continue
    
    # ----------------------- Prepare data ------------------------------------ #
    meta_cols = [c for c in ['outcome_id', 'epa_sample_id'] if c in train.columns]
    
    X_train_df = train.drop(columns=[selected_assay] + meta_cols).copy()
    X_test_df = test.drop(columns=[selected_assay] + meta_cols).copy()
    
    maccs_cols = [c for c in X_train_df.columns if c.startswith("MACCS_")]
    
    if maccs_cols:
        X_train_df[maccs_cols] = X_train_df[maccs_cols].astype("int8")
        X_test_df[maccs_cols] = X_test_df[maccs_cols].astype("int8")
    
    # ----------------------- Boruta feature selection (HTTr only) ------------ #
    httr_cols_train = [col for col in X_train_df.columns
                       if col.endswith(('_max', '_dose_at_max', '_AUC_neg'))]
    
    rf_for_boruta = RandomForestClassifier(
        n_jobs=N_JOBS,
        n_estimators=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
    
    # Stability-based feature selection over multiple random undersamples
    hits_counter = Counter()
    valid_runs = 0
    pos_idx_all = np.where(y_train == 1)[0]
    neg_idx_all = np.where(y_train == 0)[0]
    
    for r in range(STABILITY_RUNS):
        if len(neg_idx_all) > len(pos_idx_all) and len(pos_idx_all) > 0:
            rs = np.random.RandomState(RANDOM_STATE + r)
            neg_sel = rs.choice(neg_idx_all, size=len(pos_idx_all), replace=False)
            sel_idx = np.concatenate([pos_idx_all, neg_sel])
        else:
            sel_idx = np.arange(len(y_train))
        
        X_train_httr_boruta = X_train_df.iloc[sel_idx][httr_cols_train].values
        y_train_boruta = y_train[sel_idx]
        
        boruta = BorutaPy(
            estimator=rf_for_boruta,
            n_estimators='auto',
            perc=100,
            verbose=0,
            random_state=RANDOM_STATE + r
        )
        
        try:
            boruta.fit(X_train_httr_boruta, y_train_boruta)
            for f, keep in zip(httr_cols_train, boruta.support_):
                if keep:
                    hits_counter[f] += 1
            valid_runs += 1
        except Exception:
            continue
    
    # Keep features selected in >= STABILITY_MIN_FRAC of valid runs
    threshold_hits = int(np.ceil(STABILITY_MIN_FRAC * max(valid_runs, 1)))
    selected_httr_features = [f for f, h in hits_counter.items() if h >= threshold_hits]
    
    # Fallback if stability keeps nothing
    if len(selected_httr_features) == 0:
        rf_for_boruta.fit(X_train_df[httr_cols_train].values, y_train)
        importances = rf_for_boruta.feature_importances_
        order = np.argsort(importances)[::-1]
        top_k = max(25, min(200, int(np.sqrt(len(httr_cols_train)))))
        selected_httr_features = [httr_cols_train[i] for i in order[:top_k]]
    
    feature_hits.update(selected_httr_features)
    fold_selected_features.append(selected_httr_features)
    
    # ----------------------- Prepare pipeline with SMOTE + XGB -------------- #
    feature_cols = chemical_cols + selected_httr_features
    
    X_train_selected = X_train_df[feature_cols]
    X_test_selected = X_test_df[feature_cols]
    
    categorical_idx = [feature_cols.index(c) for c in feature_cols if c in maccs_cols]
    
    # Store class distribution info before SMOTE (for figures)
    pre_smote_pos = int((y_train == 1).sum())
    pre_smote_neg = int((y_train == 0).sum())
    
    base_model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        random_state=RANDOM_STATE,
        n_jobs=1,
        tree_method='hist'
    )
    
    # Create pipeline with SMOTE + XGB
    pipeline = ImbPipeline([
        ('smote', SMOTENC(
            categorical_features=categorical_idx,
            sampling_strategy='auto',
            random_state=RANDOM_STATE
        )),
        ('classifier', base_model)
    ])
    
    param_distributions = {
        'classifier__max_depth': [3, 4, 5, 6, 7, 8],
        'classifier__colsample_bytree': [0.4, 0.6, 0.8, 1.0],
        'classifier__reg_alpha': [0, 1e-3, 1e-2, 1e-1, 1, 5],
        'classifier__reg_lambda': [0.5, 1, 2, 5, 10],
        'classifier__learning_rate': [0.02, 0.05, 0.1],
        'classifier__n_estimators': [400, 600, 800, 1000, 1200]
    }
    
    # Use StratifiedGroupKFold for inner CV to prevent chemical leakage
    train_groups = train['outcome_id'].astype(str).values
    inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    tuner = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=50,
        scoring='average_precision',
        cv=inner_cv,
        verbose=0,
        n_jobs=N_JOBS,
        refit=True,
        random_state=RANDOM_STATE
    )
    
    tuner.fit(X_train_selected, y_train, groups=train_groups)
    model = tuner.best_estimator_
    
    # Get SMOTE-resampled data info for class distributions
    X_resampled_check = model.named_steps['smote'].fit_resample(X_train_selected, y_train)
    post_smote_pos = int((X_resampled_check[1] == 1).sum())
    post_smote_neg = int((X_resampled_check[1] == 0).sum())
    
    # Store class distribution info (for figures)
    fold_class_distributions.append({
        'fold': fold,
        'train_samples': len(y_train),
        'test_samples': len(y_test),
        'train_pos_pre_smote': pre_smote_pos,
        'train_neg_pre_smote': pre_smote_neg,
        'train_pos_post_smote': post_smote_pos,
        'train_neg_post_smote': post_smote_neg,
        'test_pos': int((y_test == 1).sum()),
        'test_neg': int((y_test == 0).sum()),
        'smote_applied': True
    })
    
    # ----------------------- Evaluate ---------------------------------------- #
    y_proba = model.predict_proba(X_test_selected)[:, 1]
    xgb_model = model.named_steps['classifier']
    
    # Store OOF predictions for global threshold selection
    oof_proba[test_idx] = y_proba
    oof_true[test_idx] = y_test
    
    # ----------------------- SHAP values (aggregated only) ------------------- #
    try:
        X_resampled = model.named_steps['smote'].fit_resample(X_train_selected, y_train)[0]
        X_resampled_df = pd.DataFrame(X_resampled, columns=feature_cols)
        
        explainer = shap.Explainer(
            xgb_model, 
            X_resampled_df, 
            algorithm="tree", 
            feature_names=feature_cols,
            silent=True
        )
        shap_explanation = explainer(X_test_selected, silent=True)
        shap_values_test = shap_explanation.values
        
        # Store mean absolute SHAP values for aggregation
        mean_abs_shap = np.abs(shap_values_test).mean(axis=0)
        shap_values_all_folds.append(pd.DataFrame({
            'fold': fold,
            'feature': feature_cols,
            'mean_abs_shap': mean_abs_shap
        }))
        
        # Store SHAP values for OOF aggregation (needed for violin plots)
        base_values_test = shap_explanation.base_values
        shap_cols = [f"SHAP_{c}" for c in feature_cols]
        shap_df_fold = pd.DataFrame(shap_values_test, columns=shap_cols)
        meta_df = test[IDs_TO_KEEP].copy() if IDs_TO_KEEP else pd.DataFrame(index=np.arange(len(test)))
        meta_df['fold'] = fold
        meta_df['y_true'] = y_test
        meta_df['y_proba'] = y_proba
        meta_df['base_value'] = base_values_test
        feat_vals_df = X_test_selected.reset_index(drop=True).copy()
        
        shap_out_fold = pd.concat([meta_df.reset_index(drop=True), 
                                  feat_vals_df,
                                  shap_df_fold], axis=1)
        shap_data_per_fold.append(shap_out_fold)
        
        # Optionally save detailed per-fold SHAP values
        if SAVE_DETAILED_SHAP:
            shap_dir = RUN_DIR / 'shap' / 'per_fold'
            shap_dir.mkdir(parents=True, exist_ok=True)
            shap_out_fold.to_feather(shap_dir / f'shap_test_fold_{fold}.feather')
    except Exception as e:
        print(f"[Fold {fold}] SHAP calculation failed: {e}")
    
    # ----------------------- Metrics ----------------------------------------- #
    try:
        roc = roc_auc_score(y_test, y_proba)
    except ValueError:
        roc = np.nan
    
    precision, recall, thr = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(recall, precision) if len(recall) > 1 else np.nan
    
    try:
        ap = average_precision_score(y_test, y_proba)
    except ValueError:
        ap = np.nan
    
    brier = brier_score_loss(y_test, y_proba)
    
    try:
        ll = log_loss(y_test, y_proba)
    except ValueError:
        ll = np.nan
    
    # Per-fold optimal F1 threshold
    if thr.size > 0:
        f1s = (2 * precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-12)
        best_idx = int(np.nanargmax(f1s))
        fold_opt_threshold = float(thr[best_idx])
        
        y_pred_opt = (y_proba >= fold_opt_threshold).astype(int)
        acc_opt = accuracy_score(y_test, y_pred_opt)
        f1_opt = f1_score(y_test, y_pred_opt, zero_division=0)
        cm_opt = confusion_matrix(y_test, y_pred_opt, labels=[0, 1])
    else:
        fold_opt_threshold = 0.5
        y_pred_opt = (y_proba >= 0.5).astype(int)
        acc_opt = np.nan
        f1_opt = np.nan
        cm_opt = confusion_matrix(y_test, y_pred_opt, labels=[0, 1])
    
    # Store fold metrics
    fold_metrics_detailed.append({
        'fold': fold,
        'n_train': len(y_train),
        'n_test': len(y_test),
        'train_pos': int((y_train == 1).sum()),
        'train_neg': int((y_train == 0).sum()),
        'test_pos': int((y_test == 1).sum()),
        'test_neg': int((y_test == 0).sum()),
        'roc_auc': roc,
        'pr_auc': pr_auc,
        'average_precision': ap,
        'brier_score': brier,
        'log_loss': ll,
        'optimal_threshold': fold_opt_threshold,
        'accuracy_at_optimal': acc_opt,
        'f1_at_optimal': f1_opt,
        'tn': int(cm_opt[0, 0]),
        'fp': int(cm_opt[0, 1]),
        'fn': int(cm_opt[1, 0]),
        'tp': int(cm_opt[1, 1]),
        'precision_at_optimal': int(cm_opt[1, 1]) / (int(cm_opt[1, 1]) + int(cm_opt[0, 1])) if (int(cm_opt[1, 1]) + int(cm_opt[0, 1])) > 0 else 0,
        'recall_at_optimal': int(cm_opt[1, 1]) / (int(cm_opt[1, 1]) + int(cm_opt[1, 0])) if (int(cm_opt[1, 1]) + int(cm_opt[1, 0])) > 0 else 0,
        'specificity_at_optimal': int(cm_opt[0, 0]) / (int(cm_opt[0, 0]) + int(cm_opt[0, 1])) if (int(cm_opt[0, 0]) + int(cm_opt[0, 1])) > 0 else 0,
        'httr_features_selected': len(selected_httr_features),
        'best_params': json.dumps({k.replace('classifier__', ''): v for k, v in tuner.best_params_.items() if k.startswith('classifier__')})
    })
    
    # Store feature importances
    if hasattr(xgb_model, 'feature_importances_'):
        imp_df = pd.DataFrame({
            'fold': fold,
            'feature': feature_cols,
            'importance': xgb_model.feature_importances_
        })
        fold_importances.append(imp_df)
    
    # Optionally save model
    if SAVE_MODELS:
        (RUN_DIR / 'models').mkdir(parents=True, exist_ok=True)
        joblib.dump(model, RUN_DIR / 'models' / f'model_fold_{fold}.joblib')
    
    # Print fold summary
    cmopt_str = f"[{cm_opt.ravel()[0]},{cm_opt.ravel()[1]};{cm_opt.ravel()[2]},{cm_opt.ravel()[3]}]"
    print(
        f"\033[1m[Fold {fold}/{N_SPLITS}]\033[0m "
        f"HTTr: {len(selected_httr_features):>3} | "
        f"ROC: {roc:.3f} | PR-AUC: {pr_auc:.3f} | AP: {ap:.3f} | "
        f"F1: {f1_opt:.3f} | CM: {cmopt_str}"
    )

end_all = datetime.now()


# ------------------------------ Global threshold (OOF) ----------------------- #
mask = ~np.isnan(oof_proba)
y_true_oof = oof_true[mask]
y_prob_oof = oof_proba[mask]

if len(y_true_oof) == 0:
    global_threshold = 0.5
    print("\nNo valid OOF predictions found; using default threshold 0.5")
else:
    prec_oof, rec_oof, thr_oof = precision_recall_curve(y_true_oof, y_prob_oof)
    if len(thr_oof) == 0:
        global_threshold = 0.5
        print("\nUnable to compute PR thresholds; using default threshold 0.5")
    else:
        f1s = (2 * prec_oof[:-1] * rec_oof[:-1]) / (prec_oof[:-1] + rec_oof[:-1] + 1e-12)
        best_idx = int(np.nanargmax(f1s))
        global_threshold = float(thr_oof[best_idx])
    
    y_pred_oof = (y_prob_oof >= global_threshold).astype(int)
    oof_roc = roc_auc_score(y_true_oof, y_prob_oof) if len(np.unique(y_true_oof)) > 1 else float('nan')
    oof_pr = auc(rec_oof, prec_oof) if len(rec_oof) > 1 else float('nan')
    oof_acc = accuracy_score(y_true_oof, y_pred_oof)
    oof_f1 = f1_score(y_true_oof, y_pred_oof, zero_division=0)
    oof_cm = confusion_matrix(y_true_oof, y_pred_oof, labels=[0, 1])
    
    print(f"\n{'='*70}")
    print(f"OOF Results (optimal threshold: {global_threshold:.4f})")
    print(f"{'='*70}")
    print(f"ROC-AUC: {oof_roc:.3f} | PR-AUC: {oof_pr:.3f} | "
          f"Accuracy: {oof_acc:.3f} | F1: {oof_f1:.3f}")
    print(f"Confusion Matrix: {oof_cm.tolist()}\n")


# ------------------------------ Save essential outputs ----------------------- #

# 1. Run summary (comprehensive metadata and performance metrics)
# Contains all aggregated CV results needed for results tables in the paper
if fold_metrics_detailed:
    metrics_df = pd.DataFrame(fold_metrics_detailed)
    cv_summary = {
        'target_assay': selected_assay,
        'n_folds': N_SPLITS,
        'random_state': RANDOM_STATE,
        'global_threshold': global_threshold,
        'cv_roc_auc_mean': float(metrics_df['roc_auc'].mean()),
        'cv_roc_auc_std': float(metrics_df['roc_auc'].std()),
        'cv_pr_auc_mean': float(metrics_df['pr_auc'].mean()),
        'cv_pr_auc_std': float(metrics_df['pr_auc'].std()),
        'cv_average_precision_mean': float(metrics_df['average_precision'].mean()),
        'cv_average_precision_std': float(metrics_df['average_precision'].std()),
        'cv_f1_mean': float(metrics_df['f1_at_optimal'].mean()),
        'cv_f1_std': float(metrics_df['f1_at_optimal'].std()),
        'cv_accuracy_mean': float(metrics_df['accuracy_at_optimal'].mean()),
        'cv_accuracy_std': float(metrics_df['accuracy_at_optimal'].std()),
        'cv_precision_mean': float(metrics_df['precision_at_optimal'].mean()),
        'cv_precision_std': float(metrics_df['precision_at_optimal'].std()),
        'cv_recall_mean': float(metrics_df['recall_at_optimal'].mean()),
        'cv_recall_std': float(metrics_df['recall_at_optimal'].std()),
        'cv_specificity_mean': float(metrics_df['specificity_at_optimal'].mean()),
        'cv_specificity_std': float(metrics_df['specificity_at_optimal'].std()),
        'cv_brier_score_mean': float(metrics_df['brier_score'].mean()),
        'cv_brier_score_std': float(metrics_df['brier_score'].std()),
        'cv_log_loss_mean': float(metrics_df['log_loss'].mean()),
        'cv_log_loss_std': float(metrics_df['log_loss'].std()),
        'oof_roc_auc': float(oof_roc) if 'oof_roc' in locals() and not np.isnan(oof_roc) else None,
        'oof_pr_auc': float(oof_pr) if 'oof_pr' in locals() and not np.isnan(oof_pr) else None,
        'oof_accuracy': float(oof_acc) if 'oof_acc' in locals() else None,
        'oof_f1': float(oof_f1) if 'oof_f1' in locals() else None,
        **initial_stats,
        'run_start_time': start_all.isoformat(),
        'run_end_time': end_all.isoformat(),
        'run_duration_seconds': (end_all - start_all).total_seconds()
    }
    
    with open(RUN_DIR / 'run_summary.json', 'w') as f:
        json.dump(cv_summary, f, indent=2)
    print(f"✓ Saved: run_summary.json")

# 2. Feature importance summary (aggregated across folds for main figures)
# Contains mean importance, std, and selection frequency for feature importance plots
if fold_importances:
    importances_df = pd.concat(fold_importances, axis=0, ignore_index=True)
    
    importance_stats = importances_df.groupby('feature')['importance'].agg([
        'count', 'mean', 'std', 'min', 'max'
    ]).reset_index()
    importance_stats.columns = ['feature', 'fold_count', 'importance_mean', 
                                'importance_std', 'importance_min', 'importance_max']
    importance_stats = importance_stats.sort_values('importance_mean', ascending=False)
    
    # Add feature type classification
    importance_stats['feature_type'] = importance_stats['feature'].apply(
        lambda x: 'httr' if any(x.endswith(suffix) for suffix in ['_max', '_dose_at_max', '_AUC_neg'])
                 else 'chemical' if x.startswith('MACCS_')
                 else 'other'
    )
    
    # Add Boruta selection frequency
    feat_hits_df = pd.DataFrame.from_dict(feature_hits, orient='index', columns=['selection_count']).reset_index()
    feat_hits_df.columns = ['feature', 'selection_count']
    importance_stats = importance_stats.merge(feat_hits_df, on='feature', how='left')
    importance_stats['selection_count'] = importance_stats['selection_count'].fillna(0).astype(int)
    
    importance_stats.to_csv(RUN_DIR / 'feature_importance_summary.csv', index=False)
    print(f"✓ Saved: feature_importance_summary.csv")

# 3. SHAP global ranking (aggregated across folds for interpretability figures)
# Contains mean absolute SHAP values for model interpretation plots
if shap_values_all_folds:
    shap_df = pd.concat(shap_values_all_folds, axis=0, ignore_index=True)
    
    shap_summary = shap_df.groupby('feature')['mean_abs_shap'].agg([
        'count', 'mean', 'std'
    ]).reset_index()
    shap_summary.columns = ['feature', 'fold_count', 'mean_abs_shap', 'std_abs_shap']
    shap_summary = shap_summary.sort_values('mean_abs_shap', ascending=False)
    
    # Add feature type classification
    shap_summary['feature_type'] = shap_summary['feature'].apply(
        lambda x: 'httr' if any(x.endswith(suffix) for suffix in ['_max', '_dose_at_max', '_AUC_neg'])
                 else 'chemical' if x.startswith('MACCS_')
                 else 'other'
    )
    
    (RUN_DIR / 'shap').mkdir(exist_ok=True, parents=True)
    shap_summary.to_csv(RUN_DIR / 'shap' / 'shap_global_rank.csv', index=False)
    print(f"✓ Saved: shap/shap_global_rank.csv")

print(f"\n{'='*70}")
print(f"All outputs saved to: {RUN_DIR}")
print(f"Total runtime: {(end_all - start_all).total_seconds():.1f} seconds")
print(f"{'='*70}\n")



Target: TOX21_p450_CYP2C9_Antagonist
Samples: 3,050 | Positives: 1,596 | Negatives: 1,454
 
[Fold 1/3] HTTr selected:  108 | ROC AUC: 0.907 | PR AUC: 0.919 | AP: 0.919 | Brier: 0.126 | LogLoss: 0.445 | ACC: 0.844 | F1: 0.849 | CM: [426,76;88,460] | thr: 0.558
